# TA-BN-ODE + DSTPP: Temporal Adaptive Neural ODEs with Deep Spatio-Temporal Point Processes
## Real-Time Network Intrusion Detection

**Paper:** *Temporal Adaptive Neural Ordinary Differential Equations with Deep Spatio-Temporal Point Processes for Real-Time Network Intrusion Detection*  
**Journal:** Complex and Intelligent Systems (Q1)  
**Authors:** Roger Nick Anaedevha, Alexander G. Trofimov, Yuri V. Borodachev  
**Affiliation:** National Research Nuclear University MEPhI

---

### Five Key Contributions
1. **TA-BN-ODE Architecture** — Continuous-depth networks with time-dependent normalisation (Section IV)
2. **Multi-Scale DSTPP** — Transformer-enhanced point processes with log-barrier optimisation (Section V)
3. **Structured Bayesian Inference** — PAC-Bayesian bounds, calibrated uncertainty (Section VI)
4. **LLM Zero-Shot Detection** — Temporal reasoning prompts for novel attacks (Section VII)
5. **Online Learning** — EWC + PSI drift detection with convergence guarantees (Section VIII-F)

### Datasets
- **ICS3D** (18.9M records): DOI [10.34740/kaggle/dsv/12483891](https://doi.org/10.34740/kaggle/dsv/12483891)
- **Benchmarks**: DOI [10.34740/KAGGLE/DSV/12479689](https://doi.org/10.34740/KAGGLE/DSV/12479689)

In [ ]:
import sys
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings

warnings.filterwarnings('ignore')

# Add parent dir to path
sys.path.insert(0, os.path.dirname(os.getcwd()) if 'notebooks' in os.getcwd() else os.getcwd())
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'reproducibility') if 'notebooks' not in os.getcwd() else os.path.dirname(os.getcwd()))

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

## 1. Load ICS3D Datasets

In [ ]:
from src.data.loader import ICS3DDataLoader, create_dataloaders
from src.data.preprocessing import Preprocessor

# Load Container Security dataset
loader = ICS3DDataLoader()
X, y = loader.load_containers(mode='DNN')

# Preprocess
preprocessor = Preprocessor()
X, y = preprocessor.fit_transform(X, y)

print(f'Features:  {X.shape}')
print(f'Classes:   {preprocessor.n_classes} ({preprocessor.class_names[:5]}...)')
print(f'Samples:   {len(X):,}')

In [ ]:
# Create DataLoaders (temporal split 70/15/15)
train_loader, val_loader, test_loader = create_dataloaders(
    X, y, batch_size=256, num_workers=2
)

## 2. Build Model — TA-BN-ODE + DSTPP Framework

The complete architecture (Figure 2) integrates:
- Multi-scale TA-BN-ODE (time constants: 10⁻⁶, 10⁻³, 1, 3600 s)
- Transformer-enhanced point process (8 heads, 4 layers)
- Bayesian uncertainty via MC dropout
- Coupled continuous-discrete dynamics

In [ ]:
from src.models.framework import TABNODEPointProcessFramework

model = TABNODEPointProcessFramework(
    input_dim=X.shape[1],
    hidden_dim=256,
    n_classes=preprocessor.n_classes,
    n_ode_blocks=2,
    time_constants=[1e-6, 1e-3, 1.0, 3600.0],
    n_heads=8,
    n_transformer_layers=4,
    d_model=512,
    solver='dopri5',
    rtol=1e-3,
    atol=1e-4,
    mc_samples_train=10,
    mc_samples_test=50,
)

print(model.summary())

## 3. Training

Combined loss (Eq. 3): L = L_cls + λ₁·L_TPP + λ₂·L_ELBO + λ₃·L_reg
- Adam optimiser with cosine annealing (10⁻³ → 10⁻⁵)
- Gradient clipping (max_norm=1.0)
- Early stopping (patience=10)

In [ ]:
from src.training.trainer import Trainer

trainer = Trainer(
    model, device,
    lr=1e-3, lr_min=1e-5,
    max_grad_norm=1.0,
    patience=10,
    checkpoint_dir='checkpoints/containers'
)

# Train (reduce epochs for quick demo; paper uses 100)
history = trainer.train(
    train_loader, val_loader,
    epochs=30,
    tpp_weight=0.1,
    kl_weight=1e-4,
    reg_weight=1e-3,
)

## 4. Comprehensive Evaluation

In [ ]:
from src.evaluation.metrics import Evaluator

evaluator = Evaluator(model, device)

# Detection performance (Table III)
print('=== Detection Performance ===')
detection_results = evaluator.evaluate_detection(
    test_loader,
    class_names=preprocessor.class_names.tolist()
)

In [ ]:
# Uncertainty calibration (Figure 4, Table VIII)
print('=== Uncertainty Calibration ===')
calibration_results = evaluator.evaluate_calibration(test_loader, n_mc_samples=50)

In [ ]:
# Throughput and latency (Table IX)
print('=== Throughput & Latency ===')
perf_results = evaluator.evaluate_throughput(test_loader, warmup_iters=50, measure_iters=200)

## 5. Visualisation

In [ ]:
from src.evaluation.visualization import Visualizer
import matplotlib.pyplot as plt

viz = Visualizer(save_dir='figures')
viz.plot_training_convergence(history)
viz.plot_parameter_efficiency()

## 6. Online Adaptation (Concept Drift)

EWC + PSI drift detection (Algorithm S2):
- PSI threshold = 0.2 triggers adaptation
- EMA rate ρ = 0.98
- EWC weight η = 5×10⁻³

In [ ]:
from src.training.online_adapter import OnlineAdapter
from torch.utils.data import DataLoader

adapter = OnlineAdapter(
    model, device,
    ema_rate=0.98,
    ewc_weight=5e-3,
    psi_threshold=0.2,
)

# Simulate streaming evaluation
stream_loader = DataLoader(test_loader.dataset, batch_size=32, shuffle=True)
streaming_results, streaming_acc = evaluator.evaluate_streaming(
    stream_loader, max_steps=200
)
print(f"\nStreaming Accuracy: {streaming_results['streaming_accuracy']:.4f}")

## 7. LLM Zero-Shot Detection (Section VII)

Temporal reasoning with Meta-Llama-3.1-8B-Instruct.  
Falls back to rule-based heuristics when LLM is not available.

In [ ]:
from src.models.llm_integration import LLMTemporalReasoner

reasoner = LLMTemporalReasoner()

# Demo with sample event sequence
sample_timestamps = np.array([0.0, 0.001, 0.003, 0.004, 0.01, 0.015, 0.02])
sample_features = np.random.randn(7, 8)

result = reasoner.predict(sample_timestamps, sample_features)
print(f"Threat Level: {result['threat_level']}")
print(f"Confidence:   {result['confidence']:.3f}")
print(f"Reasoning:    {result['reasoning'][:200]}")

## 8. Results Summary

In [ ]:
print('=' * 60)
print('FINAL RESULTS SUMMARY')
print('=' * 60)
print(f"Model Parameters:     {model.count_parameters():,} ({model.count_parameters()/1e6:.2f}M)")
print(f"Detection Accuracy:   {detection_results.get('accuracy', 0):.4f}")
print(f"F1 Score (weighted):  {detection_results.get('f1_weighted', 0):.4f}")
print(f"ECE:                  {calibration_results.get('ece', 0):.4f}")
print(f"Throughput:           {perf_results.get('throughput_M_events_per_sec', 0):.2f}M events/s")
print(f"P50 Latency:          {perf_results.get('p50_latency_ms', 0):.2f}ms")
print(f"Streaming Accuracy:   {streaming_results.get('streaming_accuracy', 0):.4f}")
print('=' * 60)